[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haricalzi/PhD-Lectures/blob/main/Bioinformatics/XAI_PSN.ipynb)

# Counterfactual Explanations on Patient Similarity Networks

*Hands-on lab, Bioinformatics, Università degli Studi di Milano*

**Hari Calzi** · [hari.calzi@unimi.it](mailto:hari.calzi@unimi.it)

1. Setup
2. Dataset: ROSMAP multi-omics
3. Preprocessing
4. Patient Similarity Network: one graph per omic, fused with SNF
5. A GNN to classify patients: GraphSAGE
6. Explaining a prediction: gradient saliency
7. Counterfactual explanation with CE-GNN
8. Evaluating counterfactuals: validity, fidelity, sparsity

## 1. Setup

In [ ]:
import importlib.util

if importlib.util.find_spec("torch") is None:
    %pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q torch_geometric pandas numpy matplotlib scikit-learn networkx

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn.functional as F
from scipy.spatial.distance import cdist
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import coalesce, homophily

torch.manual_seed(0)
LABELS = np.array(["NC", "AD"])
COLORS = np.array(["#4C72B0", "#C44E52"])

## 2. Dataset

**ROSMAP** (Religious Orders Study and Memory and Aging Project): post-mortem brain samples of 351 subjects, 182 with Alzheimer's disease (AD) and 169 normal controls (NC). Each subject has three omics: mRNA expression, DNA methylation and miRNA expression.

We use the version released with MOGONET (Wang et al., *Nature Communications*, 2021), the same dataset seen in the lecture: 200 pre-selected features per omic, and a fixed split into 245 training and 106 test subjects.

In [ ]:
URL = "https://raw.githubusercontent.com/txWang/MOGONET/main/ROSMAP/"
OMICS = {1: "mRNA", 2: "methylation", 3: "miRNA"}


def read(name):
    return pd.read_csv(URL + name, header=None)


views, feature_names = {}, []
for i, omic in OMICS.items():
    views[omic] = pd.concat([read(f"{i}_tr.csv"), read(f"{i}_te.csv")], ignore_index=True)
    feature_names += [f"{omic}: {f}" for f in read(f"{i}_featname.csv")[0]]

y_train, y_test = read("labels_tr.csv")[0], read("labels_te.csv")[0]
y = np.concatenate([y_train, y_test]).astype(int)
n_train = len(y_train)

print({omic: v.shape for omic, v in views.items()})
print("NC / AD:", np.bincount(y))
views["mRNA"].iloc[:5, :8]

## 3. Preprocessing

A single step: every feature is standardized (z-score), so that all features and all omics weigh the same when we compute distances between patients. Then we stack the three omics into one feature matrix.

In [ ]:
for omic, v in views.items():
    assert not v.isna().any().any(), f"missing values in {omic}"
    views[omic] = ((v - v.mean()) / v.std()).values

x = torch.tensor(np.hstack(list(views.values())), dtype=torch.float)
y_t = torch.tensor(y)
train_mask = torch.arange(len(y)) < n_train
print(x.shape)

## 4. Patient Similarity Network

For each omic we compute a patient-by-patient similarity with the scaled exponential kernel of SNF:

$$W(i,j) = \exp\left(-\frac{\rho^2(x_i, x_j)}{\mu\,\varepsilon_{ij}}\right)$$

Then SNF fuses the three matrices: each one is iteratively updated using the others, so that similarities supported by several omics get stronger. Finally, to get a graph, every patient is connected to its 10 most similar patients.

In [ ]:
def affinity(X, K=20, mu=0.5):
    dist = cdist(X, X)
    sigma = np.sort(dist, axis=1)[:, 1:K + 1].mean(axis=1)
    eps = (sigma[:, None] + sigma[None, :] + dist) / 3
    return np.exp(-dist**2 / (mu * eps))


def normalize(W):
    W = W.copy()
    np.fill_diagonal(W, 0)
    W = W / (2 * W.sum(axis=1, keepdims=True))
    np.fill_diagonal(W, 0.5)
    return (W + W.T) / 2


def keep_knn(W, K=20):
    S = np.zeros_like(W)
    rows = np.arange(len(W))[:, None]
    nn = np.argsort(-W, axis=1)[:, 1:K + 1]
    S[rows, nn] = W[rows, nn]
    return S / S.sum(axis=1, keepdims=True)


def snf(Ws, K=20, iterations=20):
    P = [normalize(W) for W in Ws]
    S = [keep_knn(W, K) for W in Ws]
    for _ in range(iterations):
        P = [normalize(S[v] @ np.mean([P[u] for u in range(len(P)) if u != v], axis=0) @ S[v].T)
             for v in range(len(P))]
    return np.mean(P, axis=0)


def knn_graph(W, k=10):
    W = W.copy()
    np.fill_diagonal(W, 0)
    src = np.repeat(np.arange(len(W)), k)
    dst = np.argsort(-W, axis=1)[:, :k].ravel()
    edges = torch.tensor(np.array([np.concatenate([src, dst]), np.concatenate([dst, src])]))
    return coalesce(edges)

In [ ]:
similarity = {omic: affinity(v) for omic, v in views.items()}
similarity["fused (SNF)"] = snf(list(similarity.values()))
graphs = {name: knn_graph(W) for name, W in similarity.items()}

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (name, edges) in zip(axes.ravel(), graphs.items()):
    G = nx.Graph(edges.t().tolist())
    pos = nx.spring_layout(G, seed=0)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.1)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=15, node_color=COLORS[y[list(G)]])
    ax.set_title(f"{name}  (homophily = {homophily(edges, y_t):.2f})")
    ax.axis("off")
fig.suptitle("Patient Similarity Networks (blue: NC, red: AD)", fontsize=14)
plt.tight_layout()
plt.show()

### Is this homophily good or bad?

**Homophily** is the fraction of edges that connect two patients with the same label:

$$h = \frac{|\{(i,j) \in E : y_i = y_j\}|}{|E|}$$

To read the number we need a reference. If edges were drawn at random, with our two almost balanced classes (169 NC, 182 AD) we would get

$$h_\text{random} = p_\text{NC}^2 + p_\text{AD}^2 \approx 0.48^2 + 0.52^2 \approx 0.50$$

So:

- $h \approx 1$: neighbours almost always share the label. The structure alone nearly solves the task (citation networks used as GNN benchmarks are around 0.8).
- $h \approx 0.5$: the graph is no better than random with respect to the label.
- $h < 0.5$: *heterophily*, neighbours tend to have the opposite label.

Our networks are between 0.52 and 0.61: **above random, but low**. The graph carries some useful signal, but a patient's neighbours are often of the other class. This is common for real omics data, where AD and NC subjects largely overlap. It has two consequences:

1. a model that simply averages over neighbours would mix AD and NC information and blur the prediction, so the patient's own data must keep its weight (next section);
2. fusion is not guaranteed to increase homophily. SNF integrates the omics into a single, more robust network; it does not optimize for the label (it never sees it).

From now on we use the fused network.

In [ ]:
edge_index = graphs["fused (SNF)"]

## 5. A GNN to classify patients: GraphSAGE

GNNs work by **message passing**: at each layer, every patient updates its representation using those of its neighbours. GraphSAGE (Hamilton et al., *NeurIPS* 2017) does it in three steps:

1. **aggregate**: average the representations of the neighbours, $\;m_i = \operatorname{mean}_{j \in \mathcal N(i)} h_j$
2. **combine**: transform the patient's own representation and the aggregated message with two *different* weight matrices and sum them
3. **activate**: apply a non-linearity (ReLU)

$$h_i' = \sigma\big(W_1\, h_i + W_2\, m_i\big)$$

Compare it with a plain GCN, where the patient is just one more element of a (normalized) average over its neighbourhood, with a single weight matrix:

$$h_i' = \sigma\Big(W \sum_{j \in \mathcal N(i) \cup \{i\}} \tfrac{1}{\sqrt{d_i d_j}}\, h_j\Big)$$

With 10 neighbours, in a GCN the patient's own omics count for roughly 1/11 of the input. In GraphSAGE, $W_1$ and $W_2$ are learned separately, so the model can decide how much to trust the patient and how much to trust its neighbours. Given the low homophily we just measured, this makes a big difference (on this data, roughly 79% vs 67% test accuracy).

With two layers, the prediction for a patient depends on its own features, on its neighbours, and on the neighbours of its neighbours (**2-hop** neighbourhood).

All 351 patients are in the graph, but only the labels of the 245 training patients are used for training (*transductive* setting).

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, 2)

    def embed(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

    def forward(self, x, edge_index):
        h = F.dropout(self.embed(x, edge_index), p=0.5, training=self.training)
        return self.conv2(h, edge_index)


model = GNN(x.size(1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    loss = F.cross_entropy(model(x, edge_index)[train_mask], y_t[train_mask])
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    proba = F.softmax(model(x, edge_index), dim=1)
pred = proba.argmax(1)

print(f"train accuracy: {(pred[train_mask] == y_t[train_mask]).float().mean():.3f}")
print(f"test accuracy:  {(pred[~train_mask] == y_t[~train_mask]).float().mean():.3f}")

## 6. Explaining a prediction: gradient saliency

We pick a test patient correctly predicted as AD and ask why.

**Saliency** is the gradient of the predicted class score with respect to the input. A large gradient means that a small change of that input would change the prediction a lot. On a graph the input is the feature matrix of *all* patients, so one gradient gives us two explanations:

- the row of our patient: which of **its omic features** matter
- the rows of the other patients: which **neighbours** matter

In [ ]:
candidates = torch.where(~train_mask & (pred == 1) & (y_t == 1))[0]
patient = int(candidates[(proba[candidates, 1] - 0.75).abs().argmin()])
print(f"patient {patient}: true = {LABELS[y[patient]]}, predicted = {LABELS[pred[patient]]} "
      f"(p = {proba[patient, 1]:.2f})")

x_in = x.clone().requires_grad_(True)
model(x_in, edge_index)[patient, pred[patient]].backward()
saliency = x_in.grad.abs()

In [ ]:
importance = pd.Series(saliency[patient].numpy(), index=feature_names)
omic_of = importance.index.str.split(":").str[0]
omic_colors = {"mRNA": "#55A868", "methylation": "#8172B2", "miRNA": "#CCB974"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 2]})
importance.groupby(omic_of).sum().plot.bar(ax=axes[0], color=[omic_colors[o] for o in sorted(omic_colors)], rot=0)
axes[0].set_title("total saliency per omic")
top = importance.nlargest(15)[::-1]
top.plot.barh(ax=axes[1], color=[omic_colors[n.split(":")[0]] for n in top.index])
axes[1].set_title(f"top 15 features, patient {patient}")
plt.tight_layout()
plt.show()

In [ ]:
G = nx.Graph(edge_index.t().tolist())
ego = nx.ego_graph(G, patient, radius=2)
node_importance = saliency.sum(1)

pos = nx.spring_layout(ego, seed=0)
sizes = 1500 * node_importance[list(ego)] / node_importance[list(ego)].max()
plt.figure(figsize=(8, 7))
nx.draw_networkx_edges(ego, pos, alpha=0.2)
nx.draw_networkx_nodes(ego, pos, node_size=sizes + 10, node_color=COLORS[y[list(ego)]])
nx.draw_networkx_nodes(ego, pos, nodelist=[patient], node_size=sizes[list(ego).index(patient)] + 10,
                       node_color=COLORS[y[patient]], edgecolors="k", linewidths=2)
plt.title(f"2-hop neighbourhood of patient {patient}: node size = saliency")
plt.axis("off")
plt.show()

## 7. Counterfactual explanation with CE-GNN

A counterfactual answers a different question: **what would have to change for the prediction to be different?**

We follow the CE-GNN approach (Chaidos, Dimitriou, Calzi et al., [ANNPR 2026](https://link.springer.com/chapter/10.1007/978-3-032-39028-8_5)), in two steps.

**Step 1: retrieval in the embedding space.** We take the hidden representation $h_i$ learned by the GNN for every patient and retrieve the patient *predicted in the opposite class* that is most similar to ours by cosine similarity:

$$\text{sim}(h_i, h_j) = \frac{h_i^\top h_j}{\|h_i\|\,\|h_j\|}$$

The counterfactual is a real patient, so it is plausible by construction.

In [ ]:
with torch.no_grad():
    h = F.normalize(model.embed(x, edge_index), dim=1)

cosine = h @ h[patient]
opposite = torch.where(pred != pred[patient])[0]
cf = int(opposite[cosine[opposite].argmax()])
print(f"patient {patient} ({LABELS[pred[patient]]})  ->  counterfactual {cf} ({LABELS[pred[cf]]}), "
      f"cosine similarity = {cosine[cf]:.2f}")

**Step 2: which features make the difference?** For every feature $k$ we compute the absolute difference between the patient $i$ and its counterfactual $c$:

$$\Delta_k = |x_{c,k} - x_{i,k}|$$

Then, starting from the feature with the largest $\Delta_k$, we copy the counterfactual's values into the patient one feature at a time, and stop as soon as the prediction flips.

In [ ]:
difference = (x[cf] - x[patient]).abs()
order = difference.argsort(descending=True)

x_cf = x.clone()
p_ad = [proba[patient, 1].item()]
with torch.no_grad():
    for n, f in enumerate(order, start=1):
        x_cf[patient, f] = x[cf, f]
        p_ad.append(F.softmax(model(x_cf, edge_index), dim=1)[patient, 1].item())
        if (p_ad[-1] > 0.5) != (pred[patient] == 1):
            break
changed = order[:n]

plt.figure(figsize=(7, 3.5))
plt.plot(p_ad, marker="o")
plt.axhline(0.5, color="k", ls="--", lw=0.8)
plt.xlabel("number of features copied from the counterfactual")
plt.ylabel("p(AD)")
plt.title(f"patient {patient}: {n} features changed")
plt.show()

pd.DataFrame({"patient": x[patient, changed].numpy(), "counterfactual": x[cf, changed].numpy()},
             index=[feature_names[f] for f in changed]).round(2)

The table is the counterfactual explanation: the features (in standardized units) that, if they had the values of the most similar NC patient, would make the model predict NC. Note that the patient keeps its neighbours in the graph: only its own omic profile changes.

## 8. Evaluating CE-GNN on all patients

We now generate a counterfactual for 10 test patients, with the same two steps as above, and evaluate them with three metrics (on the whole cohort it works the same way, it just takes a few minutes):

$$\text{Validity} = \left(\frac{\text{number of successful prediction flips}}{\text{total number of generated counterfactuals}}\right) \times 100$$

$$\text{Fidelity} = \frac{1}{|\mathcal X|}\sum_{x \in \mathcal X}\Big(\mathbb 1[\Phi(x) = y_x] - \mathbb 1[\Phi(x') = y_x]\Big)$$

$$\text{Sparsity} = \frac{1}{|\mathcal X|}\sum_{x \in \mathcal X}\left(1 - \frac{\text{number of important features}}{\text{total number of features}}\right)$$

where $\Phi$ is the GNN, $y_x$ the true label of patient $x$, $x'$ its counterfactual, and the *important features* are the ones changed to obtain it.

- **Validity**: percentage of counterfactuals that actually flip the prediction.
- **Fidelity**: accuracy on the original patients minus accuracy on their counterfactuals. If the changes remove exactly what the model needs, correct predictions become wrong and fidelity is high.
- **Sparsity**: average fraction of features left untouched. The closer to 1, the more compact the explanation.

In [ ]:
def ce_gnn(i):
    opposite = torch.where(pred != pred[i])[0]
    c = opposite[(h[opposite] @ h[i]).argmax()]
    order = (x[c] - x[i]).abs().argsort(descending=True)
    x_new = x.clone()
    with torch.no_grad():
        for n, f in enumerate(order, start=1):
            x_new[i, f] = x[c, f]
            new_pred = int(model(x_new, edge_index)[i].argmax())
            if new_pred != pred[i]:
                break
    return {"patient": i, "counterfactual": int(c), "true": LABELS[y[i]], "pred": LABELS[pred[i]],
            "cf_pred": LABELS[new_pred], "n_changed": n}


patients = torch.where(~train_mask)[0][:10].tolist()
results = pd.DataFrame([ce_gnn(i) for i in patients])

validity = (results.cf_pred != results.pred).mean() * 100
fidelity = ((results.pred == results.true).astype(int) - (results.cf_pred == results.true).astype(int)).mean()
sparsity = (1 - results.n_changed / x.size(1)).mean()

print(f"Validity: {validity:.2f}%")
print(f"Fidelity: {fidelity:.3f}")
print(f"Sparsity: {sparsity:.3f}")
results

## References

- N. Chaidos, A. Dimitriou, H. Calzi, E. Casiraghi, G. Stamou, G. Valentini. [*Counterfactual Explanations for Graph Neural Networks in Patient Outcome Prediction*](https://link.springer.com/chapter/10.1007/978-3-032-39028-8_5). Artificial Neural Networks in Pattern Recognition (ANNPR 2026), LNCS, Springer, pp. 52–64.
- T. Wang et al. [*MOGONET integrates multi-omics data using graph convolutional networks allowing patient classification and biomarker identification*](https://doi.org/10.1038/s41467-021-23774-w). Nature Communications, 2021.
- B. Wang et al. [*Similarity network fusion for aggregating data types on a genomic scale*](https://doi.org/10.1038/nmeth.2810). Nature Methods, 2014.
- W. Hamilton, R. Ying, J. Leskovec. [*Inductive Representation Learning on Large Graphs*](https://arxiv.org/abs/1706.02216) (GraphSAGE). NeurIPS, 2017.